### Seção de <font color=yellow>imports</font>
#### Nesta seção carregamos (importamos) as bibliotecas que serão usadas neste notebook

In [ ]:
import dlib
import cv2
from PIL import Image, ImageOps
import numpy as np
import matplotlib.pyplot as plt
from helper import *
import time
import re
import json


### Lendo uma imagem usando opencv e exibindo conteúdo usando matplotlib


In [ ]:
img_bgr = cv2.imread('../images/han_solo.png')
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
fig = plt.figure(figsize=(4, 4))
plt.axis('off')
fig = plt.imshow(img_rgb)


### Carregando um detector facial da biblioteca dlib

In [ ]:
dlib_face_detector = dlib.get_frontal_face_detector()
dlib_face_predictor = dlib.shape_predictor("../models/shape_predictor_68_face_landmarks.dat")


### Aqui criamos/definimos duas funções usadas neste notebook

#### <font color=yellow>detectar_faces</font>: 

- a partir de uma imagem, obtém:

    - (x1, y1, x2, y2) coordenadas do retângulo
    
    - 32 pontos (x, y) de interesse da face (face landmarks)

    - ao todo, são 68 números (34 x e 34 y), por isso o nome "68_face_landmarks"

#### <font color=yellow>exibir_destacando_faces</font>: 

- a partir de uma imagem, das coordenadas das faces e pontos de interesse (opcional):

    - gera uma nova imagem com as faces destacadas (e os pontos, case tenham sido passados)

#### <font color=yellow>Observação</font>: 

- Algumas outras funções foram definidas no arquivo helper.py



In [ ]:
def detectar_faces(img_gray, dlib_face_detector, dlib_face_predictor):
	rects = dlib_face_detector(img_gray, 0)
	coords = []
	pontos = []
	shape = None
	for (i, rect) in enumerate(rects):
		shape = dlib_face_predictor(img_gray, rect)
		(x1,y1,x2,y2) = shape.rect.left(), shape.rect.top(), shape.rect.right(), shape.rect.bottom ()
		coords.append((x1,y1,x2,y2))
		face_parts = []
		for p in shape.parts():
			face_parts.append((p.x, p.y))

		pontos.append(face_parts)

	return coords, pontos
	

def exibir_destacando_faces(img_rgb, coords, pontos=None, figsize=(4, 4)):
	img_decorada = img_rgb.copy()
	if not coords: pontos = []
	elif coords and not pontos: pontos = [[]]*len(coords)
	for i, (face_coords, face_pts) in enumerate(zip(coords, pontos)):
		(x1,y1,x2,y2) = face_coords
		cv2.rectangle(img_decorada, (x1, y1), (x2, y2), (0, 255, 0), 2)
		for (x, y) in face_pts:
			cv2.circle(img_decorada, (x, y), 2, (255, 255, 0), -1)

	fig = plt.figure(figsize=figsize)
	plt.axis('off')
	fig = plt.imshow(img_decorada)



#### Exemplo 1 

- Aqui exibimos uma imagem com as faces detectadas.

    - Observe que uma face obtida é um erro

In [ ]:
img_path = '../images/han_solo.png'
img_bgr = cv2.imread(img_path)
print("Dimensoes da imagem: ", img_bgr.shape)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
inicio = time.perf_counter()
coords, pontos = detectar_faces(img_gray, dlib_face_detector, dlib_face_predictor)
fim = time.perf_counter()
print(f"Tempo de detecção de faces dlib: {fim - inicio:.4f} segundos. Permite aprox. {1/(fim - inicio)} detecs/s.")
for _idx, c in enumerate(coords, start=1):
    print(f"Face {_idx}: ", c)

exibir_destacando_faces(img_rgb, coords, pontos, figsize=(6,4))


#### Exemplo 2

- Mais uma imagem com as faces detectadas.

- Observe o tempo de detecção de faces. Isso seria viável para vídeos?


In [ ]:
img_path = '../images/lewinsky.png'
img_bgr = cv2.imread(img_path)
print("Dimensoes da imagem: ", img_bgr.shape)
print(f"'Área' da imagem: {(img_bgr.shape[0]*img_bgr.shape[1])//1000}K px2")

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
inicio = time.perf_counter()
coords, pontos = detectar_faces(img_gray, dlib_face_detector, dlib_face_predictor)
fim = time.perf_counter()
print(f"Tempo de detecção de faces dlib: {fim - inicio:.4f} segundos. Permite aprox. {1/(fim - inicio)} detecs/s.")
img_faces = []
for _idx, fc in enumerate(coords, start=1):
    print(f"Face {_idx}: ", fc)
    (x1,y1,x2,y2) = fc
    img_faces.append(img_rgb[y1:y2, x1:x2])

show_image(img_faces, size=(2*len(img_faces), 2), axis='off', titulos=[f'face{i}' for i in range(len(img_faces))])
exibir_destacando_faces(img_rgb, coords, pontos)


#### Exemplo 3

- Mais uma imagem com as faces detectadas.
 
- Compare o tempo de detecção com a imagem anterior. Por que houve variação?

In [ ]:
img_path = '../images/epstein.png'
img_bgr = cv2.imread(img_path)
print("Dimensoes da imagem: ", img_bgr.shape)
print(f"'Área' da imagem: {(img_bgr.shape[0]*img_bgr.shape[1])//1000}K px2")

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
inicio = time.perf_counter()
coords, pontos = detectar_faces(img_gray, dlib_face_detector, dlib_face_predictor)
fim = time.perf_counter()
print(f"Tempo de detecção de faces dlib: {fim - inicio:.4f} segundos. Permite aprox. {1/(fim - inicio)} detecs/s.")
img_faces = []
for _idx, fc in enumerate(coords, start=1):
    print(f"Face {_idx}: ", fc)
    (x1,y1,x2,y2) = fc
    img_faces.append(img_rgb[y1:y2, x1:x2])

show_image(img_faces, size=(2*len(img_faces), 2), axis='off', titulos=[f'face{i}' for i in range(len(img_faces))])
exibir_destacando_faces(img_rgb, coords, pontos)



#### Exemplo 4

- Tentativa de detecção.

- Não foram detectadas as faces. Quais prováveis justificativas?


In [ ]:
img_path = '../images/revista.png'
img_bgr = cv2.imread(img_path)
print("Dimensoes da imagem: ", img_bgr.shape)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
inicio = time.perf_counter()
coords, pontos = detectar_faces(img_gray, dlib_face_detector, dlib_face_predictor)
print(coords, pontos)
fim = time.perf_counter()
print(f"Tempo de detecção de faces dlib: {fim - inicio:.4f} segundos. Permite aprox. {1/(fim - inicio)} detecs/s.")
img_faces = []
for _idx, fc in enumerate(coords, start=1):
    print(f"Face {_idx}: ", fc)
    (x1,y1,x2,y2) = fc
    img_faces.append(img_rgb[y1:y2, x1:x2])

if img_faces:
    show_image(img_faces, size=(2*len(img_faces), 2), axis='off', titulos=[f'face{i}' for i in range(len(img_faces))])
else:
    print('Não encontrou faces na imagem!')
exibir_destacando_faces(img_rgb, coords)


#### Exemplo 5

- Nova tentativa de detecção. Agora ampliando a imagem.

- Observe o códifo dentro de for _idx, fc in enumerate(coords, start=1):

    - a lista img_faces é preenchida com imagens das faces encontradas

    - as imagens são recortadas (sliced) com as coordenadas img_rgb[y1:y2, x1:x2]

    - relembrar que as dimensoes das imagens no openvc são (altura, largura, canais)

    - diferente do plano cartesiano, onde normalmente definimos um retângulo com (x1, y1) e (x2, y2), no numpy fazemos o 'recorte' com a faixa de cada dimensao y1:y2 e x1:x2

In [ ]:
# testando agora com área ampliada
img_path = '../images/revista.png'
img_bgr = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_rgb = resize_image_cv2(img_rgb, max_dimensao=900)
print("Dimensoes da imagem redimensionada: ", img_rgb.shape)
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
inicio = time.perf_counter()
coords, pontos = detectar_faces(img_gray, dlib_face_detector, dlib_face_predictor)
fim = time.perf_counter()
print(f"Tempo de detecção de faces dlib: {fim - inicio:.4f} segundos. Permite aprox. {1/(fim - inicio)} detecs/s.")
img_faces = []
for _idx, fc in enumerate(coords, start=1):
    print(f"Face {_idx}: ", fc)
    (x1,y1,x2,y2) = fc
    img_faces.append(img_rgb[y1:y2, x1:x2])

if img_faces:
    show_image(img_faces, size=(2*len(img_faces), 2), axis='off', titulos=[f'face{i+1}' for i in range(len(img_faces))])
else:
    print('Não encontrou faces na imagem!')
exibir_destacando_faces(img_rgb, coords)

#### Exemplo 6

- Repetimos o processo do exemplo 5 mas aumentamos o recorte proporcionalmente ao tamanho do retângulo detectado.

- as faces recortadas são salvas em arquivos com cv2.imwrite(f"../images/face{_idx}.png", img_face)

In [ ]:
# obtendo área maior da face e salvando
img_path = '../images/revista.png'
img_bgr = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_rgb = resize_image_cv2(img_rgb, max_dimensao=900)
print("Dimensoes da imagem redimensionada: ", img_rgb.shape)
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
inicio = time.perf_counter()
coords, pontos = detectar_faces(img_gray, dlib_face_detector, dlib_face_predictor)
fim = time.perf_counter()
print(f"Tempo de detecção de faces dlib: {fim - inicio:.4f} segundos. Permite aprox. {1/(fim - inicio)} detecs/s.")
img_faces = []
for _idx, fc in enumerate(coords, start=1):
    print(f"Face {_idx}: ", fc)
    (x1,y1,x2,y2) = fc
    delta_y = (y2-y1)*0.25
    delta_x = (x2-x1)*0.25
    x1 = int(max(x1 - delta_x, 0))
    x2 = int(min(x2 + delta_x, img_gray.shape[1]))
    y1 = int(max(y1 - delta_y, 0))
    y2 = int(min(y2 + delta_y, img_gray.shape[0]))
    print(x1,y1,x2,y2)
    img_faces.append(img_rgb[y1:y2, x1:x2])
    img_face = cv2.cvtColor(img_rgb[y1:y2, x1:x2], cv2.COLOR_RGB2BGR)
    cv2.imwrite(f"../images/face{_idx}.png", img_face)

if img_faces:
    show_image(img_faces, size=(2*len(img_faces), 2), axis='off', titulos=[f'face{i+1}' for i in range(len(img_faces))])
else:
    print('Não encontrou faces na imagem!')
exibir_destacando_faces(img_rgb, coords, pontos)


### Obtendo o cliente para um servidor vllm (ou ollama)

- o método get_cliente_openai está definido em helper.py

- com base no cliente, podemos verificar quais modelos estão sendo 'servidos'

In [ ]:
models = []
cli = get_cliente_openai('192.168.15.60') 

for model_data in cli.models.list().model_dump()['data']:
    if model_data['object'] == 'model':
        models.append(model_data['id'])

print(models)

#### Exemplo 7

- Usando o primeiro modelo disponível para responder perguntas sobre as faces gravadas em disco no exemplo 6.

- a função get_resposta_vml está definida em helper.py

In [ ]:
model = models[0]
if model not in models: model = models[0]

for _idx in range(1, len(img_faces)+1):
    img_path = f"../images/face{_idx}.png"
    # descr = get_resposta_vml(img_path, cli, model, "Qual a idade e sexo da pessoa nessa foto, supondo que estamos em março de 2026?", answer_json=False)
    descr = get_resposta_vml(img_path, cli, model, "Qual a idade e sexo da pessoa nessa foto, supondo que estamos em março de 2026?", answer_json=True)
    print(f"Face{_idx}:\n{descr}\n\n{60*'-'}\n")


#### Exemplo 8

- Agora repetimos o processo do exemplo 8 com um prompt mais elaborado.

- Interpretamos a resposta (parse do json da resposta)

- Exibimos as faces e as descrições geradas

In [ ]:

novo_prompt = """Atue como um sistema especialista em análise demográfica e reconhecimento facial. 
Sua tarefa é analisar a imagem fornecida e extrair a idade e o sexo da pessoa.

INSTRUÇÕES CRÍTICAS:
1. Se a pessoa for anônima, e isso é mais provável, forneça uma estimativa numérica baseada em traços visuais.
2. Se tiver certeza de que a pessoa é uma figura pública, mencione o nome dela nas observações.
3. A saída deve ser estritamente um objeto JSON único, sem textos introdutórios ou conclusões.

FORMATO DE RESPOSTA (EXEMPLO):
{"idade": 18, "sexo": "Feminino", "observacoes": "Aparentemente trata-se da atriz inglesa Abigail Zoe Lewis."}

PERGUNTA: Qual a idade e sexo da pessoa nessa foto, considerando que estamos em março de 2026?"""

for _idx in range(1, len(img_faces)+1):
    img_path = f"../images/face{_idx}.png"
    resp = get_resposta_vml(img_path, cli, model, novo_prompt, answer_json=True)
    # o trecho abaixo faz o parse (interpretação / processamento) da resposta
    match = re.search(r'(\{.*\})', resp, re.DOTALL)
    descr = ''
    if match:
        json_str = match.group(1)
        dados = json.loads(json_str)
        keys = list(dados.keys())
        for k in keys:
            if k.lower() not in dados:
                dados[k.lower()] = dados[k]
                del dados[k]

        for k in ('idade', 'sexo', 'observacoes'):
            if k in dados:
                descr += f"{k.title()}: {dados[k]}\n"
                del dados[k]

        for k in dados:
            descr += f"{k.title()}: {dados[k]}\n"

        descr = descr.strip()

    else:
        descr = resp 
        
    # print(f"Face{_idx}:\n{descr}\n\n{60*'-'}\n")
    fig, ax = plt.subplots(1, 2, figsize=(9, 2), gridspec_kw={'width_ratios': [1, 4]})
    img_face = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    ax[0].imshow(img_face)
    ax[0].set_title(f"Face{_idx}", fontsize=10, fontweight='bold')
    ax[0].axis('off')
    ax[1].text(0.05, 0.5, descr, fontsize=11, 
               verticalalignment='center', horizontalalignment='left', wrap=True)
    ax[1].axis('off') # Remove os eixos do lado do texto também
    plt.tight_layout()
    plt.show()


#### Exemplo 9

- Detecção de faces em uma imagem diferente (obtida no site freepick)

-  as faces são salvas com cv2.imwrite(f"../images/face_free_pick_{_idx}.png", img_face)

    - gera as imagens <font color=yellow>face_free_pick_1.png</font> a <font color=yellow>face_free_pick_4.png</font>

In [ ]:
img_path = '../images/foto_site_free_pick.png'
img_bgr = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_rgb = resize_image_cv2(img_rgb, max_dimensao=1200)
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
inicio = time.perf_counter()
coords, pontos = detectar_faces(img_gray, dlib_face_detector, dlib_face_predictor)
fim = time.perf_counter()
print(f"Tempo de detecção de faces dlib: {fim - inicio:.4f} segundos. Permite aprox. {1/(fim - inicio)} detecs/s.")
img_faces = []
for _idx, fc in enumerate(coords, start=1):
    print(f"Face {_idx}: ", fc)
    (x1,y1,x2,y2) = fc
    delta_y = (y2-y1)*0.25
    delta_x = (x2-x1)*0.25
    x1 = int(max(x1 - delta_x, 0))
    x2 = int(min(x2 + delta_x, img_gray.shape[1]))
    y1 = int(max(y1 - delta_y, 0))
    y2 = int(min(y2 + delta_y, img_gray.shape[0]))
    print(x1,y1,x2,y2)
    img_faces.append(img_rgb[y1:y2, x1:x2])
    img_face = cv2.cvtColor(img_rgb[y1:y2, x1:x2], cv2.COLOR_RGB2BGR)
    cv2.imwrite(f"../images/face_free_pick_{_idx}.png", img_face)

if img_faces:
    show_image(img_faces, size=(2*len(img_faces), 2), axis='off', titulos=[f'face{i+1}' for i in range(len(img_faces))])
else:
    print('Não encontrou faces na imagem!')
exibir_destacando_faces(img_rgb, coords, pontos)


#### Exemplo 10

- Descrevendo as faces <font color=yellow>face_free_pick_1.png</font> a <font color=yellow>face_free_pick_4.png</font>

In [ ]:
novo_prompt = """Atue como um sistema especialista em análise demográfica e reconhecimento facial. 
Sua tarefa é analisar a imagem fornecida e extrair a idade e o sexo da pessoa.

INSTRUÇÕES CRÍTICAS:
1. Se a pessoa for anônima, e isso é mais provável, forneça uma estimativa numérica baseada em traços visuais.
2. Se tiver certeza de que a pessoa é uma figura pública, mencione o nome dela nas observações.
3. A saída deve ser estritamente um objeto JSON único, sem textos introdutórios ou conclusões.

FORMATO DE RESPOSTA (EXEMPLO):
{"idade": 18, "sexo": "Feminino", "observacoes": "Aparentemente trata-se da atriz inglesa Abigail Zoe Lewis."}

PERGUNTA: Qual a idade e sexo da pessoa nessa foto, considerando que estamos em março de 2026?"""

for _idx in range(1, len(img_faces)+1):
    img_path = f"../images/face_free_pick_{_idx}.png"
    resp = get_resposta_vml(img_path, cli, model, novo_prompt, answer_json=True)
    match = re.search(r'(\{.*\})', resp, re.DOTALL)
    descr = ''
    if match:
        json_str = match.group(1)
        dados = json.loads(json_str)
        keys = list(dados.keys())
        for k in keys:
            if k.lower() not in dados:
                dados[k.lower()] = dados[k]
                del dados[k]

        for k in ('idade', 'sexo', 'observacoes'):
            if k in dados:
                descr += f"{k.title()}: {dados[k]}\n"
                del dados[k]

        for k in dados:
            descr += f"{k.title()}: {dados[k]}\n"

        descr = descr.strip()

    else:
        descr = resp 
    
    # print(f"Face{_idx}:\n{descr}\n\n{60*'-'}\n")
    fig, ax = plt.subplots(1, 2, figsize=(9, 2), gridspec_kw={'width_ratios': [1, 4]})
    img_face = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    ax[0].imshow(img_face)
    ax[0].set_title(f"Face{_idx}", fontsize=10, fontweight='bold')
    ax[0].axis('off')
    ax[1].text(0.05, 0.5, descr, fontsize=11, 
               verticalalignment='center', horizontalalignment='left', wrap=True)
    ax[1].axis('off')
    plt.tight_layout()
    plt.show()


#### Exemplo 11

- Executando uma tarefa diferente de análise facial


In [ ]:

novo_prompt = """Atue como um sistema especialista em análise de imagens. 
Sua tarefa é analisar a imagem fornecida e extrair o número de bolhas.

INSTRUÇÕES CRÍTICAS:
1. A imagem possui fundo escurecido e as bolhas claras, de tamanho variado. Algumas maiores e outras menores, com graus de aproximação variada.
2. Conte o número de bolhas encontradas na imagem.
3. A saída deve ser estritamente um objeto JSON único, sem textos introdutórios ou conclusões.

FORMATO DE RESPOSTA (EXEMPLO):
{"bolhas": n}

PERGUNTA: Qual o número de bolhas na imagem?"""

img_path = f"../images/morfologico.png"
resp = get_resposta_vml(img_path, cli, model, novo_prompt, answer_json=True)
match = re.search(r'(\{.*\})', resp, re.DOTALL)
descr = ''
if match:
    json_str = match.group(1)
    dados = json.loads(json_str)
    keys = list(dados.keys())
    for k in dados: descr += f"{k.title()}: {dados[k]}\n"
    descr = descr.strip()
else:
    descr = resp 

fig, ax = plt.subplots(1, 2, figsize=(9, 2), gridspec_kw={'width_ratios': [1, 4]})
img_rgb = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
ax[0].imshow(img_rgb)
ax[0].set_title(f"Imagem Analisada", fontsize=10, fontweight='bold')
ax[0].axis('off')
ax[1].text(0.05, 0.5, descr, fontsize=11, 
            verticalalignment='center', horizontalalignment='left', wrap=True)
ax[1].axis('off') 
plt.tight_layout()
plt.show()


#### Finalização dos experimentos

- Que lições práticas podemos extrair desses exemplos?

- Discutir sobre natureza das imagens.

- Necessidade de conhecer as imagens do seu problema: reenquadramento de princípios de análise de dados - conheça seuas dados!!

- Em aplicações muito especializadas, pode haver dependência do dispositivo de captura.

- Conceito de imagens "in the wild". Comumente referenciado em aplicações associadas a análise facial (detecção de faces, estimativa de idade, reconhecimento facial, etc.)

- Exemplo: análise de imagens de um HD - você não sabe o que vai encontrar (tamanho, resolução, condições de iluminação, formato, integridade das imagens - imagens carved)
